## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import optuna
import warnings
import gc

warnings.filterwarnings('ignore')
print("✅ Bibliothèques chargées")

✅ Bibliothèques chargées


## 2. Chargement des données

In [2]:
print("Chargement des données...")

# Chargement par chunks pour économiser la mémoire
X_train = pd.concat([chunk for chunk in pd.read_csv('X_train.csv', chunksize=50000, low_memory=False)])
y_train = pd.read_csv('y_train.csv')
X_test = pd.concat([chunk for chunk in pd.read_csv('X_test.csv', chunksize=50000, low_memory=False)])

gc.collect()

print(f"✓ X_train : {X_train.shape}")
print(f"✓ y_train : {y_train.shape}")
print(f"✓ X_test  : {X_test.shape}")

Chargement des données...
✓ X_train : (1172086, 307)
✓ y_train : (1172086, 2)
✓ X_test  : (586044, 307)


## 3. Nettoyage : Suppression colonnes > 75% NaN

In [3]:
print("Suppression des colonnes avec > 75% de NaN...")

# Calcul du pourcentage de NaN
nan_pct = (X_train.isnull().sum() / len(X_train)) * 100

# Colonnes à supprimer
cols_to_drop = nan_pct[nan_pct > 75].index.tolist()

print(f"✓ {len(cols_to_drop)} colonnes à supprimer")

# Suppression
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

# Suppression colonne ID si présente
if 'Unnamed: 0' in X_train.columns:
    X_train = X_train.drop(columns=['Unnamed: 0'])

print(f"✓ X_train : {X_train.shape}")
print(f"✓ X_test  : {X_test.shape}")

gc.collect()

Suppression des colonnes avec > 75% de NaN...
✓ 51 colonnes à supprimer
✓ X_train : (1172086, 255)
✓ X_test  : (586044, 256)


0

## 4. Préparation des données

In [4]:
print("Préparation des données...")

# Target
y = y_train.iloc[:, -1]

# Alignement des index
common_idx = X_train.index.intersection(y.index)
X_train = X_train.loc[common_idx]
y = y.loc[common_idx]

# Identification des colonnes catégorielles
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"✓ {len(cat_cols)} colonnes catégorielles")

# Remplissage NaN catégories
for col in cat_cols:
    X_train[col] = X_train[col].fillna("MISSING").astype(str)
    if col in X_test.columns:
        X_test[col] = X_test[col].fillna("MISSING").astype(str)

print(f"✓ Dataset prêt : {X_train.shape}")

Préparation des données...
✓ 3 colonnes catégorielles
✓ Dataset prêt : (1172086, 255)


## 5. Split Train/Validation

In [5]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y, test_size=0.2, random_state=42)

print(f"✓ Train : {X_tr.shape}")
print(f"✓ Val   : {X_val.shape}")

✓ Train : (937668, 255)
✓ Val   : (234418, 255)


## 6. Optuna : Tuning CatBoost (3 hyperparamètres)

In [9]:
print("\n" + "="*60)
print(" OPTUNA : TUNING CATBOOST (3 hyperparamètres) - FAST VERSION")
print("="*60)

from optuna.integration import CatBoostPruningCallback

def objective_catboost(trial):
    # Use CPU when using callbacks (GPU doesn't support custom callbacks)
    task_type = "CPU"

    # Hyperparamètres optimisés (plus petit range = beaucoup plus rapide)
    params = {
        'iterations': trial.suggest_int('iterations', 400, 1200, step=200),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.12, log=True),
        'depth': trial.suggest_int('depth', 6, 9),
        'loss_function': 'RMSE',
        'random_seed': 42,
        'verbose': False,
        'early_stopping_rounds': 50,
        'task_type': task_type,
        'thread_count': -1  # Use all CPU cores for better performance
    }
    
    # Callback Optuna pour couper les essais pourris (gains énormes)
    pruning_callback = CatBoostPruningCallback(trial, "RMSE")

    # Subsample dynamique (accélération ×2 sans vrai impact sur les rankings des params)
    # Désactivable en changeant "True" → "False".
    use_subsample = True
    if use_subsample:
        idx = np.random.choice(len(X_tr), int(len(X_tr)*0.7), replace=False)
        X_tr_sub = X_tr.iloc[idx]
        y_tr_sub = y_tr.iloc[idx]
    else:
        X_tr_sub = X_tr
        y_tr_sub = y_tr

    model = CatBoostRegressor(**params)

    model.fit(
        X_tr_sub,
        y_tr_sub,
        eval_set=(X_val, y_val),
        cat_features=cat_cols,
        callbacks=[pruning_callback]
    )
    
    y_pred = np.clip(model.predict(X_val), 0, 1000)
    return r2_score(y_val, y_pred)


# ---- OPTUNA STUDY ----
study_cat = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study_cat.optimize(objective_catboost, n_trials=15, show_progress_bar=True)

print(f"\n✓ Meilleur R² CatBoost : {study_cat.best_value:.6f}")
print("✓ Meilleurs params :")
for k, v in study_cat.best_params.items():
    print(f"  - {k}: {v}")


[I 2025-11-30 00:44:19,378] A new study created in memory with name: no-name-a20f61a4-39c5-4d9f-878f-0688162172b8



 OPTUNA : TUNING CATBOOST (3 hyperparamètres) - FAST VERSION


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2025-11-30 00:50:32,580] Trial 0 finished with value: 0.9920168497801807 and parameters: {'iterations': 600, 'learning_rate': 0.10985745201142037, 'depth': 8}. Best is trial 0 with value: 0.9920168497801807.
[I 2025-11-30 00:56:26,429] Trial 1 finished with value: 0.990354312041885 and parameters: {'iterations': 800, 'learning_rate': 0.026450634586450826, 'depth': 6}. Best is trial 0 with value: 0.9920168497801807.
[I 2025-11-30 00:56:26,429] Trial 1 finished with value: 0.990354312041885 and parameters: {'iterations': 800, 'learning_rate': 0.026450634586450826, 'depth': 6}. Best is trial 0 with value: 0.9920168497801807.
[I 2025-11-30 01:00:27,784] Trial 2 finished with value: 0.9917806171789239 and parameters: {'iterations': 400, 'learning_rate': 0.09441609766876465, 'depth': 8}. Best is trial 0 with value: 0.9920168497801807.
[I 2025-11-30 01:00:27,784] Trial 2 finished with value: 0.9917806171789239 and parameters: {'iterations': 400, 'learning_rate': 0.09441609766876465, 'depth

## 7. Optuna : Tuning XGBoost (3 hyperparamètres)

In [14]:
from xgboost import XGBRegressor
import numpy as np

print("\n" + "="*60)
print(" OPTUNA : TUNING XGBOOST (FAST VERSION)")
print("="*60)

# Encoder catégories pour XGBoost (Fait une seule fois, très bien)
# On vérifie si ça n'a pas déjà été fait pour éviter de le refaire
if 'X_tr_xgb' not in locals():
    X_tr_xgb = X_tr.copy()
    X_val_xgb = X_val.copy()
    for col in cat_cols:
        X_tr_xgb[col] = X_tr_xgb[col].astype('category').cat.codes
        X_val_xgb[col] = X_val_xgb[col].astype('category').cat.codes

def objective_xgboost(trial):
    # 1. Paramètres bornés pour la vitesse
    # max_depth > 8 fait exploser le temps de calcul exponentiellement
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 8), 
        'subsample': trial.suggest_float('subsample', 0.5, 0.9), # Accélère l'entraînement
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9), # Accélère l'entraînement
        'random_state': 42,
        'n_jobs': -1,
        'tree_method': 'hist',  # Le plus rapide sur CPU
        'early_stopping_rounds': 50
    }
    
    # 2. Subsample dynamique des données (70% du train set pour le tuning)
    # Ça suffit largement pour trouver les bons hyperparamètres
    idx = np.random.choice(len(X_tr_xgb), int(len(X_tr_xgb)*0.7), replace=False)
    X_tr_sub = X_tr_xgb.iloc[idx]
    y_tr_sub = y_tr.iloc[idx]
    
    model = XGBRegressor(**params)
    
    # 3. Fit with eval_set for early stopping
    model.fit(
        X_tr_sub, 
        y_tr_sub, 
        eval_set=[(X_val_xgb, y_val)], 
        verbose=False
    )
    
    y_pred = np.clip(model.predict(X_val_xgb), 0, 1000)
    return r2_score(y_val, y_pred)

# Création de l'étude
study_xgb = optuna.create_study(
    direction='maximize', 
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_xgb.optimize(objective_xgboost, n_trials=15, show_progress_bar=True)

print(f"\n✓ Meilleur R² XGBoost : {study_xgb.best_value:.6f}")
print(f"✓ Meilleurs params :")
for k, v in study_xgb.best_params.items():
    print(f"  - {k}: {v}")


[I 2025-11-30 02:29:06,945] A new study created in memory with name: no-name-243f92f9-240f-4d58-9758-1e36996dcbbc



 OPTUNA : TUNING XGBOOST (FAST VERSION)


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2025-11-30 02:32:14,016] Trial 0 finished with value: 0.9908680447198319 and parameters: {'n_estimators': 500, 'learning_rate': 0.1358198535217987, 'max_depth': 7, 'subsample': 0.7394633936788146, 'colsample_bytree': 0.5624074561769746}. Best is trial 0 with value: 0.9908680447198319.
[I 2025-11-30 02:34:06,208] Trial 1 finished with value: 0.9910482236744974 and parameters: {'n_estimators': 400, 'learning_rate': 0.0224831270535195, 'max_depth': 8, 'subsample': 0.7404460046972835, 'colsample_bytree': 0.7832290311184182}. Best is trial 1 with value: 0.9910482236744974.
[I 2025-11-30 02:34:06,208] Trial 1 finished with value: 0.9910482236744974 and parameters: {'n_estimators': 400, 'learning_rate': 0.0224831270535195, 'max_depth': 8, 'subsample': 0.7404460046972835, 'colsample_bytree': 0.7832290311184182}. Best is trial 1 with value: 0.9910482236744974.
[I 2025-11-30 02:35:26,869] Trial 2 finished with value: 0.9906662855220968 and parameters: {'n_estimators': 300, 'learning_rate': 0.

## 8. Entraînement final CatBoost

In [15]:
print("\nEntraînement CatBoost final...")

best_params_cat = study_cat.best_params
best_params_cat.update({
    'loss_function': 'RMSE',
    'random_seed': 42,
    'verbose': 100,
    'early_stopping_rounds': 50
})

model_cat = CatBoostRegressor(**best_params_cat)
model_cat.fit(X_train, y, cat_features=cat_cols)

print("✓ CatBoost entraîné")


Entraînement CatBoost final...
0:	learn: 109.1196835	total: 962ms	remaining: 9m 36s
0:	learn: 109.1196835	total: 962ms	remaining: 9m 36s
100:	learn: 11.5261442	total: 1m 30s	remaining: 7m 24s
100:	learn: 11.5261442	total: 1m 30s	remaining: 7m 24s
200:	learn: 10.8059969	total: 2m 38s	remaining: 5m 14s
200:	learn: 10.8059969	total: 2m 38s	remaining: 5m 14s
300:	learn: 10.4088116	total: 3m 47s	remaining: 3m 45s
300:	learn: 10.4088116	total: 3m 47s	remaining: 3m 45s
400:	learn: 10.1118940	total: 5m 9s	remaining: 2m 33s
400:	learn: 10.1118940	total: 5m 9s	remaining: 2m 33s
500:	learn: 9.8685173	total: 6m 13s	remaining: 1m 13s
500:	learn: 9.8685173	total: 6m 13s	remaining: 1m 13s
599:	learn: 9.6497877	total: 7m 22s	remaining: 0us
599:	learn: 9.6497877	total: 7m 22s	remaining: 0us
✓ CatBoost entraîné
✓ CatBoost entraîné


## 9. Entraînement final XGBoost

In [16]:
print("\nEntraînement XGBoost final...")

# Encoder pour XGBoost
X_train_xgb = X_train.copy()
for col in cat_cols:
    X_train_xgb[col] = X_train_xgb[col].astype('category').cat.codes

best_params_xgb = study_xgb.best_params
best_params_xgb.update({
    'random_state': 42,
    'n_jobs': -1,
    'tree_method': 'hist'
})

model_xgb = XGBRegressor(**best_params_xgb)
model_xgb.fit(X_train_xgb, y, verbose=False)

print("✓ XGBoost entraîné")


Entraînement XGBoost final...
✓ XGBoost entraîné
✓ XGBoost entraîné


## 10. Prédictions Ensemble (Moyenne)

In [26]:
print("\nGénération des prédictions ensemble...")

# Préparer X_test - Important: garder les mêmes colonnes que X_train
X_test_cat = X_test.copy()
X_test_xgb = X_test.copy()

# Supprimer la colonne ID si présente (comme pour X_train)
if 'Unnamed: 0' in X_test_cat.columns:
    X_test_cat = X_test_cat.drop(columns=['Unnamed: 0'])
if 'Unnamed: 0' in X_test_xgb.columns:
    X_test_xgb = X_test_xgb.drop(columns=['Unnamed: 0'])

# Traiter les colonnes catégorielles pour CatBoost (même traitement que X_train)
for col in cat_cols:
    if col in X_test_cat.columns:
        X_test_cat[col] = X_test_cat[col].fillna("MISSING").astype(str)

# Encoder les colonnes catégorielles pour XGBoost
for col in cat_cols:
    if col in X_test_xgb.columns:
        X_test_xgb[col] = X_test_xgb[col].fillna("MISSING").astype('category').cat.codes

# Prédictions - Pour CatBoost, on doit spécifier les colonnes catégorielles
from catboost import Pool
test_pool = Pool(X_test_cat, cat_features=cat_cols)
pred_cat = model_cat.predict(test_pool)
pred_xgb = model_xgb.predict(X_test_xgb)

# Ensemble (moyenne pondérée selon performance)
weight_cat = study_cat.best_value
weight_xgb = study_xgb.best_value
total_weight = weight_cat + weight_xgb

y_pred_ensemble = (pred_cat * weight_cat + pred_xgb * weight_xgb) / total_weight

# DÉNORMALISATION : Multiplier par 1000 pour obtenir des scores entre 0-1000
y_pred_ensemble = y_pred_ensemble * 1000.0
y_pred_ensemble = np.clip(y_pred_ensemble, 0, 1000)

print(f"✓ Prédictions générées et dénormalisées")
print(f"  Poids CatBoost: {weight_cat:.4f}")
print(f"  Poids XGBoost : {weight_xgb:.4f}")
print(f"  Min : {y_pred_ensemble.min():.2f}")
print(f"  Max : {y_pred_ensemble.max():.2f}")
print(f"  Mean: {y_pred_ensemble.mean():.2f}")
print(f"  Std : {y_pred_ensemble.std():.2f}")



Génération des prédictions ensemble...
✓ Prédictions générées et dénormalisées
  Poids CatBoost: 0.9922
  Poids XGBoost : 0.9917
  Min : 0.00
  Max : 1000.00
  Mean: 338.86
  Std : 378.99
✓ Prédictions générées et dénormalisées
  Poids CatBoost: 0.9922
  Poids XGBoost : 0.9917
  Min : 0.00
  Max : 1000.00
  Mean: 338.86
  Std : 378.99


## 11. Création du fichier de soumission

## 10.5. Vérification et Dénormalisation des prédictions


In [ ]:
print("🔧 CORRECTION : Dénormalisation des prédictions")
print("=" * 60)

# Vérifier l'état actuel
print(f"\nÉtat actuel des prédictions :")
print(f"  Min  : {y_pred_ensemble.min():.2f}")
print(f"  Max  : {y_pred_ensemble.max():.2f}")
print(f"  Mean : {y_pred_ensemble.mean():.2f}")

# Forcer la dénormalisation (multiplier par 1000)
print("\n⚠️  Dénormalisation forcée : multiplication par 1000")
y_pred_ensemble = y_pred_ensemble * 1000.0
y_pred_ensemble = np.clip(y_pred_ensemble, 0, 1000)

print(f"\n✅ Prédictions corrigées :")
print(f"  Min  : {y_pred_ensemble.min():.2f}")
print(f"  Max  : {y_pred_ensemble.max():.2f}")
print(f"  Mean : {y_pred_ensemble.mean():.2f}")
print(f"  Std  : {y_pred_ensemble.std():.2f}")

print("\n👉 Relancez la cellule de création du fichier de soumission !")


Vérification des échelles des données...

y_train (target original) :
  Min  : 0.00
  Max  : 814.97
  Mean : 100.00

y (utilisé pour entraînement) :
  Min  : 0.00
  Max  : 814.97
  Mean : 100.00

Prédictions actuelles (ensemble) :
  Min  : 0.00
  Max  : 3.50
  Mean : 0.37

✓ Les données sont déjà à la bonne échelle (0-1000)


In [27]:
print("\nCréation du fichier de soumission...")

# Charger X_test original pour récupérer l'ID
X_test_original = pd.read_csv('X_test.csv', usecols=[0], nrows=len(y_pred_ensemble))

submission = pd.DataFrame({
    'ID': X_test_original.iloc[:, 0],
    'MathScore': y_pred_ensemble
})

submission.to_csv('submission_ensemble.csv', index=False)

print("✅ Fichier créé : submission_ensemble.csv")
print(f"\n{submission.head()}")
print(f"\n{submission.describe()}")


Création du fichier de soumission...
✅ Fichier créé : submission_ensemble.csv

        ID    MathScore
0   412660   396.309349
1   554658     0.000000
2   937138     0.000000
3   752986  1000.000000
4  1084508   803.085118

                 ID      MathScore
count  5.860440e+05  586044.000000
mean   8.795681e+05     338.857681
std    5.075177e+05     378.991772
min    2.000000e+00       0.000000
25%    4.404968e+05       0.000000
50%    8.795360e+05     194.492720
75%    1.318993e+06     606.790672
max    1.758129e+06    1000.000000
✅ Fichier créé : submission_ensemble.csv

        ID    MathScore
0   412660   396.309349
1   554658     0.000000
2   937138     0.000000
3   752986  1000.000000
4  1084508   803.085118

                 ID      MathScore
count  5.860440e+05  586044.000000
mean   8.795681e+05     338.857681
std    5.075177e+05     378.991772
min    2.000000e+00       0.000000
25%    4.404968e+05       0.000000
50%    8.795360e+05     194.492720
75%    1.318993e+06     606.

## 12. Évaluation sur validation

In [28]:
print("\n" + "="*60)
print(" ÉVALUATION FINALE")
print("="*60)

# Prédictions sur validation
pred_cat_val = np.clip(model_cat.predict(X_val), 0, 1000)
pred_xgb_val = np.clip(model_xgb.predict(X_val_xgb), 0, 1000)
pred_ensemble_val = (pred_cat_val * weight_cat + pred_xgb_val * weight_xgb) / total_weight

# Calcul R²
r2_cat = r2_score(y_val, pred_cat_val)
r2_xgb = r2_score(y_val, pred_xgb_val)
r2_ensemble = r2_score(y_val, pred_ensemble_val)

print(f"\n📊 R² sur Validation :")
print(f"  CatBoost  : {r2_cat:.6f}")
print(f"  XGBoost   : {r2_xgb:.6f}")
print(f"  Ensemble  : {r2_ensemble:.6f}")

if r2_ensemble >= 0.78:
    print(f"\n🎉 OBJECTIF ATTEINT ! R² ≥ 0.78")
elif r2_ensemble >= 0.77:
    print(f"\n✓ Amélioration par rapport à 0.77")
    
print("\n✅ Processus terminé !")


 ÉVALUATION FINALE

📊 R² sur Validation :
  CatBoost  : 0.993683
  XGBoost   : 0.994199
  Ensemble  : 0.994145

🎉 OBJECTIF ATTEINT ! R² ≥ 0.78

✅ Processus terminé !

📊 R² sur Validation :
  CatBoost  : 0.993683
  XGBoost   : 0.994199
  Ensemble  : 0.994145

🎉 OBJECTIF ATTEINT ! R² ≥ 0.78

✅ Processus terminé !
